In [72]:
# Import libraries
import pandas as pd
import pickle
import nltk
from nltk.stem import PorterStemmer   # reduces duplicates and improves accuracy of text comparison
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [73]:
df = pd.read_csv("medicine.csv")

In [74]:
df.head()

,index,Drug_Name,Reason,Description
0,1,A CN Gel(Topical) 20gmA CN Soap 75gm,Acne,Mild to moderate acne (spots)
1,2,A Ret 0.05% Gel 20gmA Ret 0.1% Gel 20gmA Ret 0...,Acne,A RET 0.025% is a prescription medicine that i...
2,3,ACGEL CL NANO Gel 15gm,Acne,It is used to treat acne vulgaris in people 12...
3,4,ACGEL NANO Gel 15gm,Acne,It is used to treat acne vulgaris in people 12...
4,5,Acleen 1% Lotion 25ml,Acne,treat the most severe form of acne (nodular ac...


In [75]:
print(df.shape)

(9720, 4)


In [76]:
print(df.columns)

Index(['index', 'Drug_Name', 'Reason', 'Description'], dtype='object')


In [77]:
print(df.isnull().sum())

index          0
Drug_Name      0
Reason         0
Description    0
dtype: int64


In [78]:
print("Duplicate Rows:",df.duplicated().sum())

Duplicate Rows: 0


In [79]:
# Data cleaning
df.dropna(inplace = True)

In [80]:
# removes duplicate rows
df.drop_duplicates(inplace = True)

In [81]:
# remove unwanted columns 
if "index" in df.columns:
    df.drop(columns = ["index"],inplace = True)

In [82]:
df.head()

,Drug_Name,Reason,Description
0,A CN Gel(Topical) 20gmA CN Soap 75gm,Acne,Mild to moderate acne (spots)
1,A Ret 0.05% Gel 20gmA Ret 0.1% Gel 20gmA Ret 0...,Acne,A RET 0.025% is a prescription medicine that i...
2,ACGEL CL NANO Gel 15gm,Acne,It is used to treat acne vulgaris in people 12...
3,ACGEL NANO Gel 15gm,Acne,It is used to treat acne vulgaris in people 12...
4,Acleen 1% Lotion 25ml,Acne,treat the most severe form of acne (nodular ac...


In [83]:
# create tag column-> combines all the important information about medicine into a single text column
df["tags"] = (
    df["Reason"].fillna("")+" "+df["Description"].fillna("")
)

# keep only essential columns
new_df = df[["Drug_Name","tags"]].copy()

# convert text into lowercase
new_df["tags"] = new_df["tags"].str.lower().str.strip()

In [84]:
# stemming
stemmer = PorterStemmer()
def stem(text):
    words = text.split()
    stemmed_words = []
    for word in words:
        stemmed_words.append(stemmer.stem(word))
    return " ".join(stemmed_words)

# applying stemming
new_df["tags"] = new_df["tags"].apply(stem)


In [105]:
# convert text into vector 
vectorizer = CountVectorizer(
    max_features = 5000,
    stop_words = "english"
)
vectors = vectorizer.fit_transform(new_df["tags"])
print("vector shape:",vectors.shape)

vector shape: (9720, 806)


In [106]:
def recommend(medicine_name):
    medicine_name = medicine_name.lower().strip()

    medicine_list = (
        new_df["Drug_Name"].str.lower().str.strip()
    )
    matches = medicine_list[medicine_list == medicine_name]
    if matches.empty:
        print("Medicine not found!")
        return
    medicine_index = matches.index[0]

    # Generate similarity dynamically
    # No similarity.pkl required

    distances = cosine_similarity(
        vectors[medicine_index].reshape(1, -1),
        vectors
    )[0]

    medicines_list = sorted(
        list(enumerate(distances)),
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    print("\nTop 5 Recommended Medicines:\n")

    for i in medicines_list:
        print(new_df.iloc[i[0]]["Drug_Name"])

In [107]:
recommend("a cn gel(topical) 20gma cn soap 75gm")


Top 5 Recommended Medicines:

Acnedap Gel 15gm
Acnetoin 20mg Capsule 10'SAcnetoin Gel 15gm
Acnin Pimple Care Face Pack 50gm
Adapnil Gel 15gm
Alene Gel 15gm


In [108]:
with open("medicine_data.pkl","wb") as f:
    pickle.dump(new_df.to_dict(),f)
with open("vectors.pkl","wb") as f:
    pickle.dump(vectors,f)
with open("vectorizer.pkl","wb") as f:
    pickle.dump(vectorizer,f)  

In [109]:
import pandas as pd
import pickle

In [110]:
with open("medicine_data.pkl","rb") as f:
    data = pickle.load(f)
new_df_test = pd.DataFrame(data)
with open("vectors.pkl","rb") as f:
    vectors_test = pickle.load(f)

print(new_df_test.head())
print(vectors_test.shape)

                                           Drug_Name  \
0               A CN Gel(Topical) 20gmA CN Soap 75gm   
1  A Ret 0.05% Gel 20gmA Ret 0.1% Gel 20gmA Ret 0...   
2                             ACGEL CL NANO Gel 15gm   
3                                ACGEL NANO Gel 15gm   
4                              Acleen 1% Lotion 25ml   

                                                tags  
0                      acn mild to moder acn (spots)  
1  acn a ret 0.025% is a prescript medicin that i...  
2  acn it is use to treat acn vulgari in peopl 12...  
3  acn it is use to treat acn vulgari in peopl 12...  
4  acn treat the most sever form of acn (nodular ...  
(9720, 806)
